In [0]:
%python

# Stage 1: Extract Bronze (batch_1)
# Stage 2: Run Silver SCD (batch_1)
# Stage 3: Extract Bronze (batch_2)
# Stage 4: Run Silver SCD (batch_2)
# Stage 5: Extract Bronze (batch_3)
# Stage 6: Run Silver SCD (batch_3)
# Stage 7: Build Gold
# Stage 8: Summary report

In [0]:
%python

%run ../config/pipeline_config

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

print("Orchestrator ready ✓")

In [0]:
%python

pipeline_log = {
    "pipeline_start": None,
    "pipeline_end":   None,
    "stages":         {}
}

def log_stage(name, status, start, end, error=None):
    duration = round((end - start).total_seconds(), 2)
    pipeline_log["stages"][name] = {
        "status":   status,
        "duration": f"{duration}s",
        "error":    error
    }
    icon = "✅" if status == "success" else "❌"
    print(f"  {icon} {name:<35} {status:<10} {duration}s")
    if error:
        print(f"     Error: {error}")
       


In [0]:
%python

# ── Core SCD Type 2 function ──────────────────────────────────
# STUDY NOTE:
#   Parameterised function replaces 3 identical stage cells.
#   batch_date is the only thing that changes per run.
#   All logic is identical across batches — extract it once.
#   This is the "extract, don't repeat" principle in pipelines.

def run_scd_batch(batch_date: str, batch_name: str):
    # ── Read and deduplicate Bronze ───────────────────────────
    window_spec = Window.partitionBy("customer_unique_id") \
                        .orderBy(F.desc("row_id"))

    df_batch = spark.table("default.bronze_customers") \
        .filter(F.col("modified_date") == batch_date) \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")

    row_count = df_batch.count()
    print(f"  {batch_name}: {row_count:,} rows from Bronze")

    # ── Add row hash ──────────────────────────────────────────
    df_hashed = df_batch.withColumn(
        "row_hash",
        F.sha2(F.concat_ws("|",
            F.col("customer_zip_code_prefix").cast("string"),
            F.col("customer_city"),
            F.col("customer_state")
        ), 256)
    )

    # ── First run vs incremental ──────────────────────────────
    if not spark.catalog.tableExists(SILVER_SCD_TABLE):
        print(f"  {batch_name}: First run — full insert")
        df_hashed \
            .withColumn("version_number",      F.lit(1)) \
            .withColumn("customer_key",         F.monotonically_increasing_id()) \
            .withColumn("is_current",           F.lit(True)) \
            .withColumn("effective_start_date", F.col("modified_date").cast("timestamp")) \
            .withColumn("effective_end_date",   F.lit(None).cast("timestamp")) \
            .withColumn("ingestion_timestamp",  F.current_timestamp()) \
            .write.format("delta").mode("overwrite") \
            .saveAsTable(SILVER_SCD_TABLE)

        print(f"  {batch_name}: {row_count:,} rows written to Silver")

    else:
        # ── Detect changes ────────────────────────────────────
        df_silver_current = spark.table(SILVER_SCD_TABLE) \
            .filter("is_current = true")

        df_changed = df_hashed.alias("b").join(
            df_silver_current.select(
                "customer_unique_id", "row_hash", "version_number"
            ).alias("s"),
            on="customer_unique_id",
            how="inner"
        ).filter(
            F.col("b.row_hash") != F.col("s.row_hash")
        ).select("b.*", F.col("s.version_number").alias("old_version"))

        changed_count = df_changed.count()
        print(f"  {batch_name}: {changed_count} changes detected")

        if changed_count > 0:
            # Step 1: Expire old rows
            changed_ids = df_changed.select("customer_unique_id").distinct()

            DeltaTable.forName(spark, SILVER_SCD_TABLE) \
                .alias("target").merge(
                    changed_ids.alias("source"),
                    """target.customer_unique_id = source.customer_unique_id
                       AND target.is_current = true"""
                ).whenMatchedUpdate(set={
                    "is_current":         F.lit(False),
                    "effective_end_date": F.current_timestamp()
                }).execute()

            # Step 2: Insert new versions
            df_changed \
                .withColumn("version_number",      F.col("old_version") + 1) \
                .withColumn("customer_key",         F.monotonically_increasing_id()) \
                .withColumn("is_current",           F.lit(True)) \
                .withColumn("effective_start_date", F.col("modified_date").cast("timestamp")) \
                .withColumn("effective_end_date",   F.lit(None).cast("timestamp")) \
                .withColumn("ingestion_timestamp",  F.current_timestamp()) \
                .drop("old_version") \
                .write.format("delta").mode("append") \
                .saveAsTable(SILVER_SCD_TABLE)

            print(f"  {batch_name}: ✓ {changed_count} rows expired + inserted")
        else:
            print(f"  {batch_name}: No changes — Silver unchanged")

In [0]:
%python
# ── Run all stages ────────────────────────────────────────────
pipeline_log["pipeline_start"] = datetime.now()

print("=" * 55)
print(f"PIPELINE STARTED: {pipeline_log['pipeline_start']}")
print("=" * 55)

# ── Stage 1: Silver SCD for all batches ───────────────────────
batches = [
    ("2024-01-01", "batch_1"),
    ("2024-02-01", "batch_2"),
    ("2024-03-01", "batch_3"),
]

for batch_date, batch_name in batches:
    stage = f"Silver SCD {batch_name}"
    start = datetime.now()
    try:
        run_scd_batch(batch_date, batch_name)
        log_stage(stage, "success", start, datetime.now())
    except Exception as e:
        log_stage(stage, "error", start, datetime.now(), str(e))
        raise e

# ── Stage 2: Build Gold ───────────────────────────────────────
stage = "Build Gold"
start = datetime.now()
try:
    spark.table(SILVER_SCD_TABLE) \
        .filter(F.col("is_current") == True) \
        .write.format("delta").mode("overwrite") \
        .saveAsTable(GOLD_CURRENT_TABLE)

    spark.table(SILVER_SCD_TABLE) \
        .write.format("delta").mode("overwrite") \
        .saveAsTable(GOLD_HISTORY_TABLE)

    log_stage(stage, "success", start, datetime.now())
except Exception as e:
    log_stage(stage, "error", start, datetime.now(), str(e))
    raise e

In [0]:
%python
# ── Pipeline summary ──────────────────────────────────────────
pipeline_log["pipeline_end"] = datetime.now()
total = round(
    (pipeline_log["pipeline_end"] - pipeline_log["pipeline_start"]).total_seconds(), 2
)

print("\n" + "=" * 55)
print("PIPELINE COMPLETE")
print("=" * 55)
print(f"Duration  : {total}s")
print(f"Started   : {pipeline_log['pipeline_start']}")
print(f"Finished  : {pipeline_log['pipeline_end']}")
print("-" * 55)
for name, info in pipeline_log["stages"].items():
    icon = "✅" if info["status"] == "success" else "❌"
    print(f"{icon} {name:<35} {info['duration']}")
print("=" * 55)

# Layer counts
print("\nFinal Layer Counts:")
for label, table in [
    ("Bronze", "default.bronze_customers"),
    ("Silver", SILVER_SCD_TABLE),
    ("Gold Current", GOLD_CURRENT_TABLE),
    ("Gold History", GOLD_HISTORY_TABLE),
]:
    count = spark.table(table).count()
    print(f"  {label:<15} : {count:,}")